# Ground Truth Generation - GridMind (StudyGrid FAQ)

Matches lessons `02-ground-truth.md` and `03-ground-truth-batch.md`, adapted to our project:

- data comes from `../data/studygrid_faq.json` (34 documents, each with a stable `id`)
- no `course` filtering needed - our dataset only has StudyGrid documents
- uses a plain OpenAI client (structured output via `responses.parse`), separate from the
  Groq client the running app uses in `config.py`


## 1. Load the documents

Notebook lives in `evaluation/`, so the data path is one level up.


In [2]:
import sys
sys.path.insert(0, "..")

from ingest import load_faq_data

documents = load_faq_data(path="../data/studygrid_faq.json")
len(documents)


34

In [3]:
documents[0]

{'id': 1,
 'section': 'Getting Started',
 'question': 'What is StudyGrid?',
 'answer': 'StudyGrid is a mobile app that combines group chat, class materials, shared and personal to-do lists, and smart notifications — all in one place for students.'}

## 2. Structured output schema

We want the model to return a Python object, not free text.


In [4]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]


## 3. Instructions for the question generator


In [5]:
data_gen_instructions = """
You emulate a StudyGrid user trying to understand how a feature works.
Formulate 5 questions this user might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be
complete and not too short. If possible, use as few words as possible from
the record.

The output should resemble how people ask questions on a support chat.
Not too formal, not too short, not too long.

Only ask about what this specific record actually answers.
Do not introduce features, platforms, or details that are not mentioned in the record.
""".strip()


## 4. OpenAI client

This notebook uses a plain OpenAI client (`OPENAI_API_KEY` in `.env`), separate from the
Groq client the app uses at runtime. Structured output (`responses.parse`) is what we're
relying on here, and it's the officially supported path on OpenAI's API.


In [6]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)


## 5. Try it on one document


In [7]:
import json

doc = documents[0]
user_prompt = json.dumps(doc)

messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]



In [8]:
response = openai_client.responses.parse(
    model="openai/gpt-oss-120b",
    input=messages,
    text_format=Questions
)

response.output_parsed.questions


['What does StudyGrid combine into one app?',
 'Does StudyGrid include group chat and class materials?',
 'Can I keep both shared and personal to‑do lists in StudyGrid?',
 'How do smart notifications work in StudyGrid?',
 'Is StudyGrid designed specifically for students?']

## 6. Use the reusable helper

`evaluation_utils.py` sits next to this notebook.


In [9]:
from evaluation_utils import generate_structured

result, usage = generate_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)
print(usage)


['How does StudyGrid combine group chat, class materials, and to‑do lists into one place?', 'In what way do the smart notifications help me stay organized?', 'Can I collaborate with classmates on shared tasks using StudyGrid?', 'Does StudyGrid allow me to keep my personal to‑do list separate from shared ones?', 'Who is the intended audience for this mobile app?']
ResponseUsage(input_tokens=355, input_tokens_details=InputTokensDetails(cached_tokens=256), output_tokens=682, output_tokens_details=OutputTokensDetails(reasoning_tokens=589), total_tokens=1037)


In [10]:
from evaluation_utils import calc_call_cost

calc_call_cost(usage)


{'input_cost': 2.6625e-05,
 'output_cost': 0.00020459999999999999,
 'total_cost': 0.000231225}

## 7. Wrap it: one document -> ground truth records


In [13]:
from evaluation_utils import generate_structured_with_retry


In [14]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = generate_structured_with_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
        model="openai/gpt-oss-120b"   # ← الإضافة
    )

    results = []
    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [15]:
generate_ground_truth(doc)

([{'question': 'What does StudyGrid actually do for students?', 'document': 1},
  {'question': 'Which features are included in the StudyGrid app?',
   'document': 1},
  {'question': 'Is StudyGrid only for group chat or does it have other tools?',
   'document': 1},
  {'question': 'Can I manage my to‑do lists inside StudyGrid?', 'document': 1},
  {'question': 'How does StudyGrid handle notifications for me?',
   'document': 1}],
 ResponseUsage(input_tokens=355, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=313, output_tokens_details=OutputTokensDetails(reasoning_tokens=238), total_tokens=668))

## 8. Sanity check: sequential run on the first 5 documents


In [16]:
from tqdm.auto import tqdm



## 9. Full run, in parallel

34 documents, 6 workers - well under any rate limit concern.


In [17]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=1) as pool:
    gt_results  = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/34 [00:00<?, ?it/s]

In [18]:
ground_truth = []
usages = []

for records, usage in gt_results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

170

## 10. Total cost

Note: `calc_price` uses placeholder per-token prices - check `evaluation_utils.py`.


In [19]:
from evaluation_utils import calc_total_cost

calc_total_cost(usages)

0.004698975

In [20]:

calc_call_cost(usage)

{'input_cost': 2.565e-05, 'output_cost': 0.0001338, 'total_cost': 0.00015945}

In [21]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'How does StudyGrid combine group chat, class materials, and to‑do lists into one place?',
  'document': 1},
 {'question': 'In what way do the smart notifications help me stay organized?',
  'document': 1},
 {'question': 'Can I collaborate with classmates on shared tasks using StudyGrid?',
  'document': 1},
 {'question': 'Does StudyGrid allow me to keep my personal to‑do list separate from shared ones?',
  'document': 1},
 {'question': 'Who is the intended audience for this mobile app?',
  'document': 1}]

## 11. Save the ground truth set


In [22]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)
df_ground_truth.to_csv("../data/ground_truth.csv", index=False)

In [23]:
df_ground_truth.head()

,question,document
0,Can you tell me what StudyGrid actually does f...,1
1,Is StudyGrid just a chat app or does it have o...,1
2,What kind of features are included in the Stud...,1
3,Does StudyGrid handle class materials and to‑d...,1
4,How does StudyGrid keep me notified about scho...,1


In [24]:
from ingest import load_faq_data
from retriever import Retriever
from prompts import INSTRUCTIONS, USER_PROMPT_TEMPLATE

documents = load_faq_data(path="../data/studygrid_faq.json")

retriever = Retriever(
    documents,
    instructions=INSTRUCTIONS,
    prompt_template=USER_PROMPT_TEMPLATE
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [25]:
def vector_search(query):
    return retriever.search(query, num_results=5)

In [27]:
doc_id = q["document"]
results = vector_search(query=q["question"])

for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

1 == 1: True
32 == 1: False
2 == 1: False
24 == 1: False
13 == 1: False


In [28]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [29]:
from tqdm.auto import tqdm

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [30]:
ground_truth_sample = ground_truth[:15]
relevance_total_sample = compute_relevance_total(ground_truth_sample, vector_search)
relevance_total_sample

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [31]:
for i in [0, 5, 10, 20]:
    q = ground_truth[i]
    r = vector_search(query=q["question"])
    print(i, len(r))

0 5
5 5
10 5
20 5


In [32]:
relevance_total = compute_relevance_total(ground_truth, vector_search)

  0%|          | 0/170 [00:00<?, ?it/s]

In [33]:
def hit_rate(relevance):
    cnt = 0
    for line in relevance:
        if 1 in line:
            cnt = cnt + 1
    return cnt / len(relevance)

In [ ]:
##Hit Rate and MRR

In [34]:
def mrr(relevance):
    total_score = 0.0
    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break
    return total_score / len(relevance)

In [35]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [36]:
def hit_rate(relevance):
    cnt = 0
    for line in relevance:
        if 1 in line:
            cnt = cnt + 1
    return cnt / len(relevance)


def mrr(relevance):
    total_score = 0.0
    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break
    return total_score / len(relevance)


hit_rate(relevance_total), mrr(relevance_total)

(0.9705882352941176, 0.8889215686274512)